# Workshop — RAG sobre papers académicos con LangChain y NVIDIA NIM

En este workshop construirás un sistema RAG completo alimentado con dos papers fundamentales:

- **Attention Is All You Need** — Vaswani et al. (2017)
- **Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks** — Lewis et al. (2020)

El notebook descarga los PDFs desde arXiv, extrae su texto, crea embeddings y responde preguntas usando únicamente el contenido indexado.

**Stack:** MarkItDown · LangChain · NVIDIA NIM · Qdrant

| Parte | Tarea |
|---|---|
| 0 | Instalar y configurar Colab |
| 1 | Descargar y convertir PDFs con MarkItDown |
| 2 | Crear `Document` y aplicar chunking |
| 3 | Generar embeddings NVIDIA e indexar en Qdrant |
| 4 | Construir el pipeline RAG con LCEL |
| 5 | Evaluar mediante preguntas con *ground truth* |

**Modelos:**
- Embeddings: `nvidia/nemotron-3-embed-1b`
- Generación: `nvidia/nemotron-3-super-120b-a12b`


In [7]:
# Dependencias para Google Colab
%pip install -qU "markitdown[pdf]" langchain langchain-core langchain-text-splitters langchain-nvidia-ai-endpoints langchain-qdrant qdrant-client python-dotenv requests


In [8]:
import os
import getpass
from pathlib import Path
from typing import List

import requests
from dotenv import load_dotenv

load_dotenv()

try:
    from google.colab import userdata
    NVIDIA_API_KEY = userdata.get("NVIDIA_API_KEY")
except (ImportError, KeyError):
    NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")
%pip install -qU "markitdown[pdf]" langchain langchain-core langchain-text-splitters langchain-nvidia-ai-endpoints langchain-qdrant qdrant-client python-dotenv requests

import os, getpass, requests
from pathlib import Path
from dotenv import load_dotenv
from google.colab import userdata
from markitdown import MarkItDown

try:
    NVIDIA_API_KEY = userdata.get("NVIDIA_API_KEY")
except (ImportError, KeyError):
    NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")

if not NVIDIA_API_KEY:
    NVIDIA_API_KEY = getpass.getpass("NVIDIA_API_KEY: ").strip()

os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY
GENERATION_MODEL = "nvidia/nemotron-3-ultra-550b-a55b"
EMBEDDING_MODEL = "nvidia/nemotron-3-embed-1b"

# 2. Descarga y conversión de PDFs
DOCS_DIR = Path("papers")
DOCS_DIR.mkdir(parents=True, exist_ok=True)
PAPERS = {
    "attention_paper.pdf": "https://arxiv.org/pdf/1706.03762",
    "rag_paper.pdf": "https://arxiv.org/pdf/2005.11401",
}

for name, url in PAPERS.items():
    dest = DOCS_DIR / name
    if not dest.exists():
        resp = requests.get(url, timeout=90)
        dest.write_bytes(resp.content)

md = MarkItDown(enable_plugins=False)
attention_text = md.convert(str(DOCS_DIR / "attention_paper.pdf")).text_content
rag_text = md.convert(str(DOCS_DIR / "rag_paper.pdf")).text_content
print(f"✅ PDFs cargados: Attention ({len(attention_text)} chars), RAG ({len(rag_text)} chars)")

if not NVIDIA_API_KEY:
    NVIDIA_API_KEY = getpass.getpass("NVIDIA_API_KEY: ").strip()
if not NVIDIA_API_KEY:
    raise ValueError("Falta NVIDIA_API_KEY. Créala en https://build.nvidia.com/")

os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY
GENERATION_MODEL = "nvidia/nemotron-3-super-120b-a12b"
EMBEDDING_MODEL = "nvidia/nemotron-3-embed-1b"

print("✅ Configuración NVIDIA NIM lista")


✅ PDFs cargados: Attention (40174 chars), RAG (69121 chars)
✅ Configuración NVIDIA NIM lista


---
## Parte 1 — Descargar y convertir los PDFs

La celda descarga los artículos únicamente si todavía no existen en `papers/`. Después, MarkItDown convierte cada PDF a texto conservando buena parte de su estructura.

> Si una sesión de Colab se reinicia, los archivos temporales se pierden y se descargarán nuevamente.


In [9]:
# --- Celda 3: Descarga y conversión de PDFs con MarkItDown ---
from markitdown import MarkItDown

DOCS_DIR = Path("papers")
DOCS_DIR.mkdir(parents=True, exist_ok=True)

PAPERS = {
    "attention_paper.pdf": "https://arxiv.org/pdf/1706.03762",
    "rag_paper.pdf": "https://arxiv.org/pdf/2005.11401",
}

def download_file(url: str, destination: Path) -> None:
    if destination.exists() and destination.stat().st_size > 10_000:
        print(f"↪ Ya existe: {destination.name}")
        return
    response = requests.get(url, timeout=90)
    response.raise_for_status()
    destination.write_bytes(response.content)
    print(f"⬇ Descargado: {destination.name}")

for filename, url in PAPERS.items():
    download_file(url, DOCS_DIR / filename)

md_converter = MarkItDown(enable_plugins=False)

def load_pdf(path: str | Path) -> str:
    path = Path(path)
    result = md_converter.convert(str(path))
    return result.text_content

attention_text = load_pdf(DOCS_DIR / "attention_paper.pdf")
rag_text = load_pdf(DOCS_DIR / "rag_paper.pdf")

print(f"✅ attention_paper.pdf → {len(attention_text):,} caracteres")
print(f"✅ rag_paper.pdf       → {len(rag_text):,} caracteres")

↪ Ya existe: attention_paper.pdf
↪ Ya existe: rag_paper.pdf
✅ attention_paper.pdf → 40,174 caracteres
✅ rag_paper.pdf       → 69,121 caracteres


---
## Parte 2 — Convertir a `Document` y aplicar chunking

LangChain utiliza objetos `Document` con dos campos principales:

- `page_content`: contenido textual del fragmento.
- `metadata`: título, archivo de origen e índice del fragmento.

Usaremos fragmentos de 900 caracteres con solapamiento de 120. Es un punto de partida didáctico; no es un valor universal.


In [10]:
# --- Celda 4: Chunking y Creación de Documentos ---
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

def pdf_to_documents(text: str, title: str, source: str) -> List[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=900,
        chunk_overlap=120,
        separators=["\n\n", "\n", ".", " "],
    )
    documents = []
    for index, chunk in enumerate(splitter.split_text(text)):
        if len(chunk.strip()) < 30:
            continue
        documents.append(Document(
            page_content=chunk,
            metadata={"title": title, "source": source, "chunk_index": index},
        ))
    return documents

attention_docs = pdf_to_documents(attention_text, "Attention Is All You Need", "attention_paper.pdf")
rag_docs = pdf_to_documents(rag_text, "Retrieval-Augmented Generation", "rag_paper.pdf")
all_docs = attention_docs + rag_docs

print(f"Attention → {len(attention_docs)} chunks | RAG → {len(rag_docs)} chunks | Total → {len(all_docs)} chunks")

Attention → 55 chunks | RAG → 95 chunks | Total → 150 chunks


---
## Parte 3 — Embeddings NVIDIA e índice Qdrant

`NVIDIAEmbeddings` distingue automáticamente entre embeddings de documentos y consultas mediante `embed_documents()` y `embed_query()`. La dimensión se obtiene directamente del modelo para evitar valores codificados manualmente.

Qdrant se ejecuta en memoria; al reiniciar el runtime se reconstruye el índice.


In [ ]:
# --- Celda 5: Indexación en Qdrant ---
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore

COLLECTION = "academic_papers_nvidia"

embeddings = NVIDIAEmbeddings(model=EMBEDDING_MODEL, truncate="END")
EMBEDDING_DIM = len(embeddings.embed_query("dimension probe"))

qdrant_client = QdrantClient(":memory:")
qdrant_client.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE),
)

vectorstore = QdrantVectorStore(
    client=qdrant_client,
    collection_name=COLLECTION,
    embedding=embeddings,
)
ids = vectorstore.add_documents(all_docs)
print(f"✅ {len(ids)} chunks indexados en Qdrant")

/usr/local/lib/python3.13/dist-packages/langchain_nvidia_ai_endpoints/_common.py:269: UserWarning: Found nvidia/nemotron-3-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(


---
## Parte 4 — Pipeline RAG con LCEL

El pipeline conecta recuperación, construcción del prompt, generación con Nemotron y extracción del texto final. El prompt obliga al modelo a reconocer cuando el contexto recuperado no contiene la respuesta.


In [ ]:
# --- Celda 6: Construcción de la cadena RAG con LCEL ---
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatNVIDIA(
    model=GENERATION_MODEL,
    temperature=0.2,
    top_p=0.95,
    max_completion_tokens=800,
)

retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 5, "score_threshold": 0.25},
)

def format_docs(documents: List[Document]) -> str:
    return "\n\n---\n\n".join(
        f"[{doc.metadata.get('title', 'Untitled')} | {doc.metadata.get('source', '')}]\n{doc.page_content}"
        for doc in documents
    )

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Answer using only the supplied academic-paper excerpts. "
        "If the excerpts do not contain enough evidence, say so explicitly. "
        "Answer in the same language as the question and be concise. "
        "Mention the paper title supporting the answer.",
    ),
    ("human", "CONTEXT:\n{context}\n\nQUESTION:\n{question}"),
])

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ Pipeline RAG configurado correctamente")

In [ ]:
# --- Celda nueva para el ejercicio B2 ---
def evaluar_configuracion(docs, embedding_model, llm, preguntas, chunk_size, chunk_overlap, k, score_threshold):
    # 1. Re-chunking
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    chunks = text_splitter.split_documents(docs)

    # 2. Vectorstore temporal en memoria
    temp_client = QdrantClient(":memory:")
    temp_client.create_collection(
        collection_name="eval_temp",
        vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE)
    )
    vstore = QdrantVectorStore(client=temp_client, collection_name="eval_temp", embedding=embedding_model)
    vstore.add_documents(chunks)

    # 3. Retriever con umbral adaptado
    temp_retriever = vstore.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={"k": k, "score_threshold": (score_threshold + 1) / 2}
    )

    # 4. Conteo de aciertos (fragmentos recuperados)
    aciertos = 0
    for q in preguntas:
        res = temp_retriever.invoke(q)
        if len(res) > 0:
            aciertos += 1

    return len(chunks), aciertos

# Preguntas de prueba Q1 a Q5
preguntas_eval = [
    "What BLEU score did the Transformer (big model) achieve on WMT 2014 English-to-German?",
    "How does RAG combine parametric and non-parametric memory?",
    "What is the purpose of the scaling factor sqrt(dk) in Scaled Dot-Product Attention?",
    "What are the two RAG model variants proposed in the paper and how do they differ?",
    "How many attention heads does the base Transformer model use, and what are the key and value dimensions per head?"
]

print("✅ Función auxiliar evaluar_configuracion lista para correr experimentos.")

In [ ]:
# --- Celda nueva para el ejercicio B3 ---
preguntas_trampa = [
    "What BLEU score did GPT-4 achieve when compared to the Transformer architecture?",
    "What learning rate was used to fine-tune Llama 3 in the RAG paper experiments?",
    "What is the top-1 accuracy achieved by the Transformer model on the ImageNet dataset?"
]

print("=== EVALUACIÓN DE PREGUNTAS TRAMPA (B3) ===")
for idx, q in enumerate(preguntas_trampa, 1):
    respuesta = rag_chain.invoke(q)
    print(f"\nPregunta Trampa #{idx}: {q}")
    print(f"Respuesta del RAG: {respuesta}")

In [ ]:
# --- Celda nueva para el ejercicio B3 ---
preguntas_trampa = [
    "What BLEU score did GPT-4 achieve when compared to the Transformer architecture?",
    "What learning rate was used to fine-tune Llama 3 in the RAG paper experiments?",
    "What is the top-1 accuracy achieved by the Transformer model on the ImageNet dataset?"
]

print("=== EVALUACIÓN DE PREGUNTAS TRAMPA (B3) ===")
for idx, q in enumerate(preguntas_trampa, 1):
    respuesta = rag_chain.invoke(q)
    print(f"\nPregunta Trampa #{idx}: {q}")
    print(f"Respuesta del RAG: {respuesta}")

In [ ]:
# --- Celda nueva para el ejercicio B4 ---
preguntas_es = [
    "¿Qué puntaje BLEU logró el modelo Transformer grande en WMT 2014 Inglés-a-Alemán?",
    "¿Cuál es el propósito del factor de escala raíz cuadrada de dk en la Atención Producto Punto Escalada?",
    "¿Cuáles son las dos variantes del modelo RAG propuestas en el artículo y cómo se diferencian?"
]

print("=== EVALUACIÓN MULTILINGÜE EN ESPAÑOL (B4) ===")
for q in preguntas_es:
    retrieved_chunks = retriever.invoke(q)
    ans = rag_chain.invoke(q)
    print(f"\nPregunta ES: {q}")
    print(f"Chunks recuperados: {len(retrieved_chunks)}")
    print(f"Respuesta: {ans}")

Bloque 5 (NUEVO · Ejercicio B4) · Evaluación Cross-Lingual (Español vs Inglés)
Ejecuta la comparación entre consultas en español y el corpus en inglés.

In [ ]:
# --- Celda nueva para el ejercicio B5 ---
import IPython.display as display

custom_quiz = [
    {
        "id": "Custom_Q1",
        "question": "What primary positional encoding method is used in the Transformer model and why?",
        "ground_truth": "Sine and cosine functions of different frequencies are used to allow the model to easily learn to attend by relative positions. (Section 3.5 — Attention Is All You Need)"
    },
    {
        "id": "Custom_Q2",
        "question": "What external document collection and indexing method was used for non-parametric memory in RAG?",
        "ground_truth": "A Wikipedia dump indexed using Dense Passage Retrieval (DPR) and FAISS with Maximum Inner Product Search (MIPS). (Section 2.1 — RAG Paper)"
    },
    {
        "id": "Custom_Q3",
        "question": "How do both the Transformer and RAG architectures address scalability limits in processing long sequence dependencies?",
        "ground_truth": "Transformer eliminates recurrence in favor of parallel self-attention, while RAG connects parametric representations with an external non-parametric retrieval memory."
    }
]

print("=== QUIZ PERSONALIZADO B5 ===")
for item in custom_quiz:
    ans = rag_chain.invoke(item["question"])
    print(f"\n📌 [{item['id']}] Pregunta: {item['question']}")
    print(f"Respuesta RAG: {ans}\n")
    display.display(display.HTML(f"""
    <details>
    <summary>✅ Ground Truth — {item['id']}</summary>
    <p><b>Respuesta esperada:</b> {item['ground_truth']}</p>
    </details>
    <br>
    """))

---
## Parte 5 — Quiz sobre los papers

Ejecuta cada pregunta y compara la respuesta del RAG con el *ground truth*. Las preguntas están en inglés para coincidir con el idioma de los artículos; el prompt pide responder en el mismo idioma de la consulta.

Además de la corrección factual, observa si el contexto recuperado proviene del paper adecuado.


In [ ]:
# ── Pregunta 1 ────────────────────────────────────────────────
q1 = "What BLEU score did the Transformer (big model) achieve on WMT 2014 English-to-German?"

answer1 = rag_chain.invoke(q1)
print(f"Q: {q1}")
print(f"A: {answer1}")

<details>
<summary>✅ Ground Truth — Pregunta 1</summary>

El Transformer (big model) obtuvo un BLEU score de **28.4** en la tarea de
traducción WMT 2014 English-to-German, superando los mejores resultados
previos (incluyendo ensembles) por más de 2 puntos BLEU.
*(Fuente: Abstract y Section 6.1 — Attention Is All You Need)*
</details>

In [ ]:
# ── Pregunta 2 ────────────────────────────────────────────────
q2 = "How does RAG combine parametric and non-parametric memory?"

answer2 = rag_chain.invoke(q2)
print(f"Q: {q2}")
print(f"A: {answer2}")

<details>
<summary>✅ Ground Truth — Pregunta 2</summary>

RAG combina **memoria paramétrica** (un modelo seq2seq pre-entrenado, BART-large)
con **memoria no-paramétrica** (un índice denso de vectores de Wikipedia).
El retriever (DPR) recupera los top-K documentos relevantes para la query,
y el generador (BART) los usa como contexto adicional para producir la respuesta.
Ambos componentes se entrenan end-to-end de forma conjunta.
*(Fuente: Abstract y Section 2 — RAG paper)*
</details>

In [ ]:
# ── Pregunta 3 ────────────────────────────────────────────────
q3 = "What is the purpose of the scaling factor sqrt(dk) in Scaled Dot-Product Attention?"

answer3 = rag_chain.invoke(q3)
print(f"Q: {q3}")
print(f"A: {answer3}")

<details>
<summary>✅ Ground Truth — Pregunta 3</summary>

El factor de escala **1/√dk** se aplica para contrarrestar el efecto de que,
con valores grandes de dk, los productos punto crecen en magnitud y empujan
la función softmax hacia regiones con gradientes muy pequeños.
Esto haría el entrenamiento inestable. La división por √dk mantiene los
productos punto en un rango con gradientes útiles.
*(Fuente: Section 3.2.1 — Attention Is All You Need)*
</details>

In [ ]:
# ── Pregunta 4 ────────────────────────────────────────────────
q4 = "What are the two RAG model variants proposed in the paper and how do they differ?"

answer4 = rag_chain.invoke(q4)
print(f"Q: {q4}")
print(f"A: {answer4}")

<details>
<summary>✅ Ground Truth — Pregunta 4</summary>

Los dos variantes son:
- **RAG-Sequence**: usa el **mismo documento recuperado** para generar toda
  la secuencia de salida. Trata el documento como una variable latente única.
- **RAG-Token**: puede usar un **documento diferente para cada token** generado,
  permitiendo combinar información de múltiples fuentes en una sola respuesta.

RAG-Token es más flexible pero más costoso computacionalmente.
*(Fuente: Section 2.1 — RAG paper)*
</details>

In [ ]:
# ── Pregunta 5 ────────────────────────────────────────────────
q5 = "How many attention heads does the base Transformer model use, and what are the key and value dimensions per head?"

answer5 = rag_chain.invoke(q5)
print(f"Q: {q5}")
print(f"A: {answer5}")

<details>
<summary>✅ Ground Truth — Pregunta 5</summary>

El Transformer base usa **h = 8** attention heads en paralelo.
Para cada cabeza, las dimensiones son **dk = dv = dmodel/h = 512/8 = 64**.
El costo computacional total es similar al de single-head attention con
dimensionalidad completa, gracias a las dimensiones reducidas por cabeza.
*(Fuente: Section 3.2.2 — Attention Is All You Need)*
</details>

---
## Bonus — Pregunta entre ambos papers

Esta pregunta exige conectar información de los dos artículos. Además de generar la respuesta, inspeccionaremos los títulos y archivos de los fragmentos recuperados.


In [ ]:
q_bonus = ("The RAG paper uses FAISS for fast retrieval with an HNSW approximation. "
           "The Attention paper proposes the Transformer architecture. "
           "What seq2seq model does RAG use as its generator component, "
           "and is it based on the Transformer?")

answer_bonus = rag_chain.invoke(q_bonus)

# Mostrar también qué chunks recuperó
docs_retrieved = retriever.invoke(q_bonus)
sources = [(d.metadata["title"], d.metadata["source"]) for d in docs_retrieved]

print(f"Q: {q_bonus}\n")
print(f"A: {answer_bonus}\n")
print(f"Fuentes recuperadas:")
for title, source in sources:
    print(f"  [{title}] — {source}")

<details>
<summary>✅ Ground Truth — Bonus</summary>

RAG usa **BART-large** como generador (Section 2.3 del RAG paper).
BART es un modelo seq2seq pre-entrenado basado en la arquitectura **Transformer**
(referencia [58] en el RAG paper = el paper "Attention Is All You Need").
El RAG paper cita explícitamente: "BART-large, a pre-trained seq2seq transformer".

Esta pregunta demuestra el valor de RAG: el sistema conecta información de
dos documentos distintos para construir una respuesta completa.
</details>

---
## Reflexión final

1. **¿Qué ocurrió cuando el contexto no contenía información suficiente?** ¿El modelo reconoció la limitación o inventó una respuesta?
2. **¿Qué fragmentos recuperó para la pregunta bonus?** ¿Aparecieron ambos papers?
3. **¿Qué efecto tendría modificar `chunk_size`, `chunk_overlap`, `k` o `score_threshold`?**
4. **¿Por qué conviene separar el modelo de embeddings del modelo generativo?**
5. **¿Qué cambiarías para llevar este prototipo a producción?** Considera persistencia, control de acceso, evaluación, observabilidad y costos.
